<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/02_FastAPI_Experimental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# NOTEBOOK 2
# FASTAPI + RAG + TinyLlama + LoRA + ngrok
# ETAPA 10
# Proyecto:
# /content/drive/MyDrive/Mx.Oscar.Hdez
# Objetivo:
# Publicar el sistema RAG como API y aplicación web experimental.
# =====================================================================
# =====================================================================
# INSTALACIÓN
# =====================================================================
!pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    peft \
    requests
# =====================================================================
# GOOGLE DRIVE
# =====================================================================
from google.colab import drive, userdata
drive.mount(
    "/content/drive",
    force_remount=False
)
# =====================================================================
# IMPORTACIONES
# =====================================================================
from pathlib import Path
from pyngrok import ngrok
import json
import multiprocessing
import os
import requests
import subprocess
import time
import torch
# =====================================================================
# 0. CONFIGURACIÓN GENERAL
# =====================================================================
# cpu  = obliga a trabajar con CPU
# cuda = obliga a trabajar con GPU
# auto = usa GPU si está disponible
# =====================================================================
DEVICE_MODE = "cpu"       # "cpu" | "cuda" | "auto"
USE_LORA = True
RUN_LOCAL_TEST = True
OPEN_NGROK = True
TOP_K = 2
# Parámetros según dispositivo
MAX_INPUT_CPU = 1000
MAX_NEW_CPU = 120
MAX_INPUT_GPU = 1650
MAX_NEW_GPU = 180
# Pregunta utilizada para comprobar que el RAG funciona
TEST_QUESTION = (
    "¿Cuál es el perfil de egreso "
    "de Ingeniería en Sistemas Computacionales?"
)
# =====================================================================
# RUTAS DEL PROYECTO
# =====================================================================
BASE_DIR = Path(
    "/content/drive/MyDrive/Mx.Oscar.Hdez"
)
DATA_DIR = (
    BASE_DIR /
    "data"
)
INDEX_DIR = (
    BASE_DIR /
    "index"
)
MODELS_DIR = (
    BASE_DIR /
    "models"
)
API_DIR = (
    BASE_DIR /
    "api"
)
LORA_DIR = (
    MODELS_DIR /
    "tinyllama_isc_lora"
)
FAISS_FILE = (
    INDEX_DIR /
    "faiss_e5.index"
)
CHUNKS_FILE = (
    DATA_DIR /
    "chunks.json"
)
MAIN_FILE = (
    API_DIR /
    "main.py"
)
LOG_FILE = (
    API_DIR /
    "uvicorn.log"
)
API_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# =====================================================================
# CONFIGURAR CPU / GPU
# =====================================================================
if DEVICE_MODE not in {
    "cpu",
    "cuda",
    "auto"
}:
    raise ValueError(
        "DEVICE_MODE debe ser "
        "'cpu', 'cuda' o 'auto'."
    )
USE_CUDA = (
    torch.cuda.is_available()
    and DEVICE_MODE != "cpu"
)
if (
    DEVICE_MODE == "cuda"
    and not torch.cuda.is_available()
):
    raise RuntimeError(
        "Se solicitó CUDA, "
        "pero la GPU no está disponible."
    )
DEVICE = (
    "cuda"
    if USE_CUDA
    else "cpu"
)
if USE_CUDA:
    MAX_INPUT_TOKENS = MAX_INPUT_GPU
    MAX_NEW_TOKENS = MAX_NEW_GPU
else:
    MAX_INPUT_TOKENS = MAX_INPUT_CPU
    MAX_NEW_TOKENS = MAX_NEW_CPU
print("=" * 65)
print(
    "DISPOSITIVO:",
    DEVICE
)
if USE_CUDA:
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print(
        "CPU lógicos:",
        multiprocessing.cpu_count()
    )
print(
    "TOP_K:",
    TOP_K
)
print(
    "Entrada máxima:",
    MAX_INPUT_TOKENS
)
print(
    "Respuesta máxima:",
    MAX_NEW_TOKENS
)
print("=" * 65)
# =====================================================================
# VERIFICAR ARCHIVOS GENERADOS POR NOTEBOOK 1
# =====================================================================
for archivo in [
    FAISS_FILE,
    CHUNKS_FILE
]:
    if not archivo.exists():
        raise FileNotFoundError(
            f"No existe {archivo}. "
            "Ejecuta primero el Notebook 1."
        )
print(
    "✓ FAISS y chunks encontrados"
)
adapter_file = (
    LORA_DIR /
    "adapter_config.json"
)
print(
    "Adaptador LoRA:",
    adapter_file.exists()
)
# =====================================================================
# SECRETS DE COLAB
# =====================================================================
# HF_TOKEN es opcional.
# NGROK_AUTHTOKEN es obligatorio si OPEN_NGROK=True.
# =====================================================================
try:
    HF_TOKEN = userdata.get(
        "HF_TOKEN"
    )
except Exception:
    HF_TOKEN = None
try:
    NGROK_TOKEN = userdata.get(
        "NGROK_AUTHTOKEN"
    )
except Exception:
    NGROK_TOKEN = None
if OPEN_NGROK and not NGROK_TOKEN:
    raise RuntimeError(
        "No se encontró NGROK_AUTHTOKEN "
        "en Colab Secrets."
    )
print(
    "HF_TOKEN disponible:",
    bool(HF_TOKEN)
)
print(
    "NGROK_AUTHTOKEN disponible:",
    bool(NGROK_TOKEN)
)
# =====================================================================
# ETAPA 10.1
# CREAR LA APLICACIÓN FASTAPI
# =====================================================================
#
# main.py realizará:
#
# FAISS + E5
#     ↓
# recuperar contexto
#     ↓
# TinyLlama
#     ↓
# respuesta RAG
#     ↓
# FastAPI
#
# =====================================================================
MAIN_CODE = r'''
from pathlib import Path
from threading import Lock
import gc
import json
import os
import faiss
import numpy as np
import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
# =====================================================================
# CONFIGURACIÓN
# =====================================================================
BASE_DIR = Path(
    os.environ[
        "RAG_BASE_DIR"
    ]
)
FAISS_FILE = (
    BASE_DIR /
    "index" /
    "faiss_e5.index"
)
CHUNKS_FILE = (
    BASE_DIR /
    "data" /
    "chunks.json"
)
LORA_DIR = (
    BASE_DIR /
    "models" /
    "tinyllama_isc_lora"
)
EMBED_MODEL = (
    "intfloat/"
    "multilingual-e5-small"
)
LLM_MODEL = (
    "TinyLlama/"
    "TinyLlama-1.1B-Chat-v1.0"
)
DEVICE_MODE = os.environ.get(
    "RAG_DEVICE",
    "auto"
).lower()
USE_LORA = (
    os.environ.get(
        "RAG_USE_LORA",
        "1"
    )
    == "1"
)
TOP_K = int(
    os.environ.get(
        "RAG_TOP_K",
        "2"
    )
)
USE_CUDA = (
    torch.cuda.is_available()
    and DEVICE_MODE != "cpu"
)
if (
    DEVICE_MODE == "cuda"
    and not torch.cuda.is_available()
):
    raise RuntimeError(
        "CUDA fue solicitado "
        "pero no está disponible."
    )
DEVICE = (
    "cuda"
    if USE_CUDA
    else "cpu"
)
if USE_CUDA:
    MAX_INPUT_TOKENS = int(
        os.environ.get(
            "RAG_MAX_INPUT_GPU",
            "1650"
        )
    )
    MAX_NEW_TOKENS = int(
        os.environ.get(
            "RAG_MAX_NEW_GPU",
            "180"
        )
    )
else:
    MAX_INPUT_TOKENS = int(
        os.environ.get(
            "RAG_MAX_INPUT_CPU",
            "1000"
        )
    )
    MAX_NEW_TOKENS = int(
        os.environ.get(
            "RAG_MAX_NEW_CPU",
            "120"
        )
    )
    torch.set_num_threads(
        max(
            1,
            os.cpu_count() or 1
        )
    )
print(
    f"Device: {DEVICE} | "
    f"TOP_K: {TOP_K} | "
    f"INPUT: {MAX_INPUT_TOKENS} | "
    f"OUTPUT: {MAX_NEW_TOKENS}",
    flush=True
)
# =====================================================================
# CARGAR FAISS Y CHUNKS
# =====================================================================
print(
    "Cargando FAISS...",
    flush=True
)
index = faiss.read_index(
    str(
        FAISS_FILE
    )
)
chunks = json.loads(
    CHUNKS_FILE.read_text(
        encoding="utf-8"
    )
)
if index.ntotal != len(
    chunks
):
    raise RuntimeError(
        "FAISS y chunks.json "
        "no contienen la misma "
        "cantidad de elementos."
    )
print(
    f"✓ FAISS listo: "
    f"{index.ntotal} vectores",
    flush=True
)
# =====================================================================
# CARGAR MODELO DE EMBEDDINGS
# =====================================================================
# E5 se mantiene en CPU.
# =====================================================================
print(
    "Cargando E5 en CPU...",
    flush=True
)
embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device="cpu"
)
embedding_model.max_seq_length = 512
print(
    "✓ E5 listo",
    flush=True
)
# =====================================================================
# CARGAR TINYLLAMA
# =====================================================================
# CPU  -> FP32
# GPU  -> FP16
# =====================================================================
print(
    "Cargando TinyLlama...",
    flush=True
)
tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )
tokenizer.truncation_side = "left"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if USE_CUDA:
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    model = model.to(
        "cuda"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    model = model.to(
        "cpu"
    )
model.config.pad_token_id = (
    tokenizer.pad_token_id
)
model.config.use_cache = True
# =====================================================================
# CARGAR LoRA SI EXISTE
# =====================================================================
lora_active = False
if (
    USE_LORA
    and (
        LORA_DIR /
        "adapter_config.json"
    ).exists()
):
    try:
        model = PeftModel.from_pretrained(
            model,
            str(
                LORA_DIR
            ),
            is_trainable=False
        )
        lora_active = True
        print(
            "✓ LoRA cargado",
            flush=True
        )
    except Exception as error:
        print(
            "⚠ LoRA no pudo cargarse:",
            error,
            flush=True
        )
model.eval()
MODEL_NAME = (
    "TinyLlama + LoRA + RAG"
    if lora_active
    else "TinyLlama + RAG"
)
print(
    "✓ Modelo activo:",
    MODEL_NAME,
    "|",
    DEVICE,
    flush=True
)
# =====================================================================
# PROMPT DEL SISTEMA
# =====================================================================
SYSTEM_PROMPT = """
Eres un asistente académico del Tecnológico Nacional de México.
Responde siempre en español.
Usa únicamente la información incluida en el CONTEXTO.
No inventes datos.
Si el contexto no contiene información suficiente, indícalo claramente.
Responde de manera clara, precisa y suficientemente desarrollada.
""".strip()
# Evita dos generaciones simultáneas.
generation_lock = Lock()
# =====================================================================
# RETRIEVER
# =====================================================================
#
# Pregunta
#    ↓
# embedding E5
#    ↓
# FAISS
#    ↓
# chunks relevantes
#
# =====================================================================
def recuperar_contexto(
    pregunta,
    top_k=TOP_K
):
    query = (
        "query: "
        + pregunta
    )
    vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    vector = np.asarray(
        vector,
        dtype="float32"
    )
    scores, indices = index.search(
        vector,
        min(
            top_k,
            index.ntotal
        )
    )
    resultados = []
    for score, posicion in zip(
        scores[0],
        indices[0]
    ):
        if posicion < 0:
            continue
        resultado = dict(
            chunks[
                posicion
            ]
        )
        resultado[
            "score"
        ] = float(
            score
        )
        resultados.append(
            resultado
        )
    return resultados
# =====================================================================
# CONSTRUIR PROMPT
# =====================================================================
def crear_prompt(
    pregunta,
    resultados
):
    contexto = "\n\n---\n\n".join(
        f"""
DOCUMENTO: {r["documento"]}
PÁGINA: {r["pagina"]}
{r["texto"]}
""".strip()
        for r in resultados
    )
    mensajes = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
CONTEXTO:
{contexto}
PREGUNTA:
{pregunta}
""".strip()
        }
    ]
    return tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )
# =====================================================================
# LIMITAR EL CONTEXTO
# =====================================================================
def limitar_contexto(
    pregunta,
    resultados
):
    seleccionados = []
    for resultado in resultados:
        candidatos = (
            seleccionados
            + [resultado]
        )
        prompt = crear_prompt(
            pregunta,
            candidatos
        )
        cantidad_tokens = len(
            tokenizer(
                prompt,
                add_special_tokens=False
            )[
                "input_ids"
            ]
        )
        if (
            cantidad_tokens
            <= MAX_INPUT_TOKENS
        ):
            seleccionados.append(
                resultado
            )
        else:
            break
    if not seleccionados:
        seleccionados = (
            resultados[:1]
        )
    return seleccionados
# =====================================================================
# GENERAR RESPUESTA RAG
# =====================================================================
def responder_rag(
    pregunta,
    top_k=TOP_K
):
    resultados = recuperar_contexto(
        pregunta,
        top_k
    )
    resultados = limitar_contexto(
        pregunta,
        resultados
    )
    if not resultados:
        return (
            "No se encontró "
            "información suficiente.",
            []
        )
    prompt = crear_prompt(
        pregunta,
        resultados
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False
    )
    inputs = {
        clave: valor.to(
            DEVICE
        )
        for clave, valor in inputs.items()
    }
    tokens_entrada = inputs[
        "input_ids"
    ].shape[-1]
    max_context = getattr(
        model.config,
        "max_position_embeddings",
        2048
    )
    with generation_lock:
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_length=min(
                    tokens_entrada
                    + MAX_NEW_TOKENS,
                    max_context
                ),
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
    respuesta = tokenizer.decode(
        outputs[0][
            tokens_entrada:
        ],
        skip_special_tokens=True
    ).strip()
    if not respuesta:
        respuesta = (
            "El modelo no generó "
            "una respuesta."
        )
    fuentes = [
        {
            "documento": r.get(
                "documento",
                ""
            ),
            "pagina": int(
                r.get(
                    "pagina",
                    0
                ) or 0
            ),
            "score": round(
                r[
                    "score"
                ],
                4
            )
        }
        for r in resultados
    ]
    return (
        respuesta,
        fuentes
    )
# =====================================================================
# FASTAPI
# =====================================================================
app = FastAPI(
    title="Asistente RAG ISC",
    version="4.0"
)
class PreguntaRequest(
    BaseModel
):
    pregunta: str = Field(
        ...,
        min_length=3
    )
    top_k: int = Field(
        default=TOP_K,
        ge=1,
        le=5
    )
# =====================================================================
# ENDPOINT /health
# =====================================================================
@app.get(
    "/health"
)
def health():
    return {
        "status": "ok",
        "modelo": MODEL_NAME,
        "device": DEVICE,
        "lora": lora_active,
        "vectores_faiss": index.ntotal,
        "top_k": TOP_K,
        "max_input": MAX_INPUT_TOKENS,
        "max_new": MAX_NEW_TOKENS
    }
# =====================================================================
# ENDPOINT /ask
# =====================================================================
@app.post(
    "/ask"
)
def ask(
    request: PreguntaRequest
):
    try:
        respuesta, fuentes = responder_rag(
            request.pregunta.strip(),
            request.top_k
        )
        return {
            "pregunta": request.pregunta,
            "respuesta": respuesta,
            "modelo": MODEL_NAME,
            "fuentes": fuentes
        }
    except Exception as error:
        print(
            "ERROR /ask:",
            repr(
                error
            ),
            flush=True
        )
        raise HTTPException(
            status_code=500,
            detail=str(
                error
            )
        )
# =====================================================================
# INTERFAZ WEB
# =====================================================================
HTML_APP = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Asistente RAG ISC</title>
<style>
body{font-family:Arial;background:#f4f6f8;margin:0;color:#202124}
.container{max-width:720px;margin:auto;padding:18px}
.card{background:#fff;padding:22px;border-radius:16px;box-shadow:0 4px 18px #0002}
textarea{width:100%;min-height:110px;padding:12px;font-size:16px;box-sizing:border-box;border:1px solid #bbb;border-radius:9px}
button{width:100%;margin-top:10px;padding:13px;border:0;border-radius:9px;background:#174ea6;color:#fff;font-size:16px;font-weight:bold}
button:disabled{opacity:.6}
.status,.source{margin-top:10px;padding:9px;background:#eef2f7;border-radius:7px;font-size:13px}
#answer{white-space:pre-wrap;line-height:1.5;margin-top:18px}
</style>
</head>
<body>
<div class="container">
<div class="card">
<h2>Asistente RAG ISC</h2>
<div id="health" class="status">Comprobando servidor...</div>
<textarea id="question" placeholder="Escribe tu pregunta..."></textarea>
<button id="button" onclick="consultar()">Consultar</button>
<div id="state"></div>
<div id="answer"></div>
<div id="sources"></div>
</div>
</div>
<script>
async function cargarHealth(){
    try{
        const response=await fetch("/health");
        const data=await response.json();
        document.getElementById("health").textContent=
            data.modelo+" | "+
            data.device.toUpperCase()+" | "+
            data.vectores_faiss+
            " vectores | TOP_K "+
            data.top_k;
    }catch(error){
        document.getElementById("health").textContent=
            "Servidor no disponible";
    }
}
async function consultar(){
    const question=document.getElementById("question").value.trim();
    if(!question)return;
    const button=document.getElementById("button");
    const state=document.getElementById("state");
    const answer=document.getElementById("answer");
    const sources=document.getElementById("sources");
    button.disabled=true;
    state.textContent="Consultando documentos...";
    answer.textContent="";
    sources.innerHTML="";
    const controller=new AbortController();
    const timeout=setTimeout(
        ()=>controller.abort(),
        300000
    );
    try{
        const response=await fetch(
            "/ask",
            {
                method:"POST",
                headers:{
                    "Content-Type":"application/json"
                },
                body:JSON.stringify({
                    pregunta:question
                }),
                signal:controller.signal
            }
        );
        clearTimeout(timeout);
        const text=await response.text();
        let data;
        try{
            data=JSON.parse(text);
        }catch{
            throw new Error(
                text || "Respuesta inválida"
            );
        }
        if(!response.ok){
            throw new Error(
                data.detail || "Error del servidor"
            );
        }
        answer.textContent=data.respuesta;
        state.textContent=
            "Respuesta generada con "+
            data.modelo;
        if(data.fuentes){
            for(const source of data.fuentes){
                const item=document.createElement("div");
                item.className="source";
                item.textContent=
                    source.documento+
                    " — pág. "+
                    source.pagina+
                    " — score "+
                    source.score;
                sources.appendChild(item);
            }
        }
    }catch(error){
        clearTimeout(timeout);
        if(error.name==="AbortError"){
            answer.textContent=
                "La consulta excedió el tiempo máximo.";
        }else{
            answer.textContent=
                "Error: "+error.message;
        }
        state.textContent=
            "Consulta no completada.";
    }finally{
        button.disabled=false;
    }
}
document.getElementById("question").addEventListener(
    "keydown",
    event=>{
        if(
            event.key==="Enter"
            && !event.shiftKey
        ){
            event.preventDefault();
            consultar();
        }
    }
);
cargarHealth();
</script>
</body>
</html>
"""
@app.get(
    "/",
    response_class=HTMLResponse
)
def home():
    return HTML_APP
'''
# =====================================================================
# GUARDAR main.py
# =====================================================================
MAIN_FILE.write_text(
    MAIN_CODE,
    encoding="utf-8"
)
print(
    "✓ main.py creado:",
    MAIN_FILE
)
# =====================================================================
# ETAPA 10.2
# DETENER INSTANCIAS ANTERIORES
# =====================================================================
#
# Se cierran servidores Uvicorn y túneles ngrok anteriores
# para evitar conflictos con el puerto 8000.
#
# =====================================================================
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(
    [
        "pkill",
        "-f",
        "uvicorn.*main:app"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(
    2
)
# =====================================================================
# VALIDAR SINTAXIS DE main.py
# =====================================================================
revision = subprocess.run(
    [
        "python",
        "-m",
        "py_compile",
        str(
            MAIN_FILE
        )
    ],
    capture_output=True,
    text=True
)
if revision.returncode != 0:
    raise RuntimeError(
        "Error de sintaxis en main.py:\n"
        + revision.stderr
    )
print(
    "✓ Sintaxis de main.py validada"
)
# =====================================================================
# ETAPA 10.3
# INICIAR FASTAPI
# =====================================================================
#
# Notebook
#    ↓
# Uvicorn
#    ↓
# main.py
#    ↓
# FastAPI en localhost:8000
#
# =====================================================================
if LOG_FILE.exists():
    LOG_FILE.unlink()
env = os.environ.copy()
env[
    "RAG_BASE_DIR"
] = str(
    BASE_DIR
)
env[
    "RAG_DEVICE"
] = DEVICE_MODE
env[
    "RAG_USE_LORA"
] = (
    "1"
    if USE_LORA
    else "0"
)
env[
    "RAG_TOP_K"
] = str(
    TOP_K
)
env[
    "RAG_MAX_INPUT_CPU"
] = str(
    MAX_INPUT_CPU
)
env[
    "RAG_MAX_NEW_CPU"
] = str(
    MAX_NEW_CPU
)
env[
    "RAG_MAX_INPUT_GPU"
] = str(
    MAX_INPUT_GPU
)
env[
    "RAG_MAX_NEW_GPU"
] = str(
    MAX_NEW_GPU
)
env[
    "PYTHONUNBUFFERED"
] = "1"
env[
    "TOKENIZERS_PARALLELISM"
] = "false"
if HF_TOKEN:
    env[
        "HF_TOKEN"
    ] = HF_TOKEN
if DEVICE == "cpu":
    cpu_count = max(
        1,
        multiprocessing.cpu_count()
    )
    env[
        "OMP_NUM_THREADS"
    ] = str(
        cpu_count
    )
    env[
        "MKL_NUM_THREADS"
    ] = str(
        cpu_count
    )
log_handle = open(
    LOG_FILE,
    "w",
    buffering=1
)
server = subprocess.Popen(
    [
        "python",
        "-u",
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
        "--workers",
        "1"
    ],
    cwd=str(
        API_DIR
    ),
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT
)
print(
    "Uvicorn PID:",
    server.pid
)
# =====================================================================
# ETAPA 10.4
# ESPERAR A QUE FASTAPI ESTÉ LISTO
# =====================================================================
LOCAL_URL = (
    "http://127.0.0.1:8000"
)
servidor_listo = False
for intento in range(
    300
):
    if (
        server.poll()
        is not None
    ):
        break
    try:
        respuesta = requests.get(
            LOCAL_URL + "/health",
            timeout=3
        )
        if respuesta.status_code == 200:
            servidor_listo = True
            break
    except Exception:
        pass
    if intento % 10 == 0:
        print(
            f"Cargando servidor... "
            f"{intento * 2}s"
        )
    time.sleep(
        2
    )
if not servidor_listo:
    log_text = (
        LOG_FILE.read_text(
            encoding="utf-8",
            errors="ignore"
        )
        if LOG_FILE.exists()
        else ""
    )
    print(
        log_text[
            -12000:
        ]
    )
    raise RuntimeError(
        "FastAPI no pudo iniciar."
    )
health = requests.get(
    LOCAL_URL + "/health",
    timeout=10
).json()
print()
print("=" * 65)
print(
    "FASTAPI ACTIVO"
)
print("=" * 65)
print(
    "Modelo:",
    health[
        "modelo"
    ]
)
print(
    "Dispositivo:",
    health[
        "device"
    ]
)
print(
    "LoRA:",
    health[
        "lora"
    ]
)
print(
    "Vectores FAISS:",
    health[
        "vectores_faiss"
    ]
)
print(
    "TOP_K:",
    health[
        "top_k"
    ]
)
print(
    "Entrada máxima:",
    health[
        "max_input"
    ]
)
print(
    "Respuesta máxima:",
    health[
        "max_new"
    ]
)
# =====================================================================
# ETAPA 10.5
# AUTOPRUEBA LOCAL
# =====================================================================
#
# Antes de crear el túnel ngrok se verifica:
#
# FastAPI
#   ↓
# Retriever
#   ↓
# TinyLlama
#   ↓
# respuesta válida
#
# =====================================================================
if RUN_LOCAL_TEST:
    print()
    print("=" * 65)
    print(
        "AUTOPRUEBA LOCAL DEL RAG"
    )
    print("=" * 65)
    inicio = time.time()
    try:
        respuesta = requests.post(
            LOCAL_URL + "/ask",
            json={
                "pregunta": TEST_QUESTION,
                "top_k": TOP_K
            },
            timeout=(
                300
                if DEVICE == "cpu"
                else 120
            )
        )
    except Exception as error:
        print(
            LOG_FILE.read_text(
                encoding="utf-8",
                errors="ignore"
            )[
                -12000:
            ]
        )
        raise RuntimeError(
            "La autoprueba no respondió: "
            f"{error}"
        )
    tiempo = (
        time.time()
        - inicio
    )
    print(
        "Status:",
        respuesta.status_code
    )
    print(
        "Tiempo:",
        round(
            tiempo,
            2
        ),
        "segundos"
    )
    if respuesta.status_code != 200:
        print(
            respuesta.text
        )
        print(
            LOG_FILE.read_text(
                encoding="utf-8",
                errors="ignore"
            )[
                -12000:
            ]
        )
        raise RuntimeError(
            "La autoprueba RAG falló."
        )
    resultado = respuesta.json()
    print(
        "✓ RAG local funcionando"
    )
    print()
    print(
        "RESPUESTA DE PRUEBA:"
    )
    print(
        resultado[
            "respuesta"
        ]
    )
    print()
    print(
        "FUENTES:"
    )
    for fuente in resultado[
        "fuentes"
    ]:
        print(
            "-",
            fuente[
                "documento"
            ],
            "| pág.",
            fuente[
                "pagina"
            ],
            "| score",
            fuente[
                "score"
            ]
        )
# =====================================================================
# ETAPA 10.6
# CREAR TÚNEL NGROK
# =====================================================================
#
# localhost:8000
#       ↓
# ngrok
#       ↓
# URL pública HTTPS
#       ↓
# celular / navegador
#
# =====================================================================
PUBLIC_URL = None
if OPEN_NGROK:
    ngrok.set_auth_token(
        NGROK_TOKEN
    )
    tunnel = ngrok.connect(
        8000
    )
    PUBLIC_URL = (
        tunnel.public_url
    )
    try:
        respuesta_publica = requests.get(
            PUBLIC_URL + "/health",
            timeout=30
        )
        print(
            "Health público:",
            respuesta_publica.status_code
        )
    except Exception as error:
        print(
            "Advertencia ngrok:",
            error
        )
    print()
    print("=" * 65)
    print(
        "APP RAG ISC ACTIVA"
    )
    print("=" * 65)
    print(
        "APP:"
    )
    print(
        PUBLIC_URL
    )
    print()
    print(
        "API Docs:"
    )
    print(
        PUBLIC_URL + "/docs"
    )
    print()
    print(
        "Health:"
    )
    print(
        PUBLIC_URL + "/health"
    )
    print()
    print(
        "Log:"
    )
    print(
        LOG_FILE
    )
    print("=" * 65)
    print(
        "El Notebook debe permanecer "
        "conectado para mantener activa la APP."
    )
else:
    print(
        "FastAPI disponible localmente en:",
        LOCAL_URL
    )